# 19 · Why parallelism: a 70B model does not fit one GPU
Level L4 · Part 3 · core · ~15 min · builds on 03

## Goal
By the end of this lesson you can show, with arithmetic from the model's size, that a 70B model does not fit on one 24 GB L4 in FP16, FP8 or 4-bit weights. You can also find the smallest tensor-parallel split that fits, and say what that split costs.

## Why it matters
- **Without it:** you try to start the model on one GPU. The engine stops while loading the weights, with an out-of-memory error, and you have to guess which card to rent next.
- **With it:** the parameter count, the bytes per parameter and the GPU memory give you the number of GPUs in one minute. The weights of the 70B model alone are 131 GiB in FP16. One L4 can use about 20.6 GiB.
- **When you don't need it:** an 8B model in FP16 (about 15 GiB of weights) fits on one L4 with room for the cache (lesson 05). You only split a model when its weights, or its cache, are larger than one GPU's usable memory.

**Without parallelism** (one L4 holds everything):
```mermaid
flowchart TB
  W1["131 GiB of FP16 weights"] --> X["one L4: 20.6 GiB usable"]
  X --> F["does not fit: load fails"]
```

**With parallelism** (eight L4s hold one copy):
```mermaid
flowchart TB
  W8["131 GiB of FP16 weights"] --> G["8 L4s: 16.4 GiB of weights each"]
  G --> R["all-reduce on every layer, then one answer"]
```

## The idea
A **weight** is one number of the model. A 70B model has about 70.6 billion of them, so the weights take 70.6 billion times the bytes per number. Tensor parallelism splits each weight matrix into equal **shards**, one per GPU, so each GPU holds a slice and computes its part. After each layer the GPUs add their partial results together with an **all-reduce**. Think of one long shelf of books cut into pieces and spread over eight desks: every desk reads its piece, and the desks pass their totals around. The split must divide the head counts evenly, because a GPU gets whole heads (step 5). So the planner allows degrees 1, 2, 4 or 8 here.

Each desk's total is incomplete: it holds only its slice, so it cannot finish the layer alone. The all-reduce adds the partials so every GPU gets the full answer.

## Picture
One model copy, split into shards and joined by an all-reduce after each layer:
```mermaid
flowchart LR
  M["70B weights: 141 GB in FP16"] -->|"split by TP"| S1["GPU 1: shard"]
  M --> S2["GPU 2: shard"]
  M --> S8["GPU 8: shard"]
  S1 --> A["all-reduce after each layer"]
  S2 --> A
  S8 --> A
  A --> O["one token out"]
```

## Already know this?
This is the first lesson of Part 3 (quantization, parallelism and hardware). If you can answer these five without looking, skip to lesson 22. Lessons 20 and 21 cover quantization, which lesson 22 does not need; come back to them when you reach the weight formats.

PCIe is the ordinary slot link between GPUs. NVLink is a much faster direct GPU-to-GPU link that L4s do not have.

1. How many bytes does one FP16 parameter take, and how big is a 70B model's weight set in GiB?
2. What share of a 24 GB L4 does vLLM use by default, and why is that not all free for the cache?
3. Why must a tensor-parallel degree divide the head counts?
4. What does an all-reduce do, and why does it cost time between GPUs with no NVLink?
5. How is a replica different from a shard?

## Step by step
### Step 1: the numbers we need
These are copies of public numbers, not files from this course. The parameter count is the Hugging Face API count for `meta-llama/Llama-3.1-70B`. The layer, head and head-size numbers are from its `config.json`, the same ones lesson 03 uses. The L4 has 24 GB (NVIDIA's datasheet). vLLM's default `gpu_memory_utilization` is 0.92, so it keeps 8% of the card free.

In [1]:
import math

GIB = 2**30                    # bytes in one GiB
L4_BYTES = 24e9                # NVIDIA L4: 24 GB
UTIL = 0.92                    # vLLM default gpu_memory_utilization
L4_USABLE = L4_BYTES * UTIL    # what one L4 gives the engine

PARAMS_70B = 70_553_706_496    # parameters of Llama 3.1 70B
LAYERS, Q_HEADS, KV_HEADS, HEAD_DIM = 80, 64, 8, 128
# Bytes of KV cache per token, all GPUs together (same formula as lesson 03):
#   first 2     = one K and one V vector
#   LAYERS      = 80 transformer layers, each with its own cache
#   KV_HEADS    = 8 key/value heads per layer
#   HEAD_DIM    = 128 numbers per head
#   last 2      = bytes per FP16 number
KV_PER_TOKEN = 2 * LAYERS * KV_HEADS * HEAD_DIM * 2

print(f"one L4 usable for vLLM: {L4_USABLE / GIB:.2f} GiB")
print(f"70B KV cache: {KV_PER_TOKEN / 1024:.0f} KiB per token (FP16)")

one L4 usable for vLLM: 20.56 GiB
70B KV cache: 320 KiB per token (FP16)


**What you should see:** one L4 gives the engine 20.56 GiB, not 24. The 70B cache costs 320 KiB per token, the same number as lesson 03.

### Step 2: the weights in three formats
A parameter takes 2 bytes in FP16, 1 byte in FP8 and about half a byte in 4-bit formats. The 4-bit row is an idealised 0.5 bytes; real 4-bit files keep a few scale numbers, so they are a little larger (lesson 20 measures this).

In [2]:
BYTES_PER_PARAM = {"FP16": 2, "FP8": 1, "4-bit": 0.5}

for fmt, b in BYTES_PER_PARAM.items():
    weights = PARAMS_70B * b
    print(f"{fmt:6} weights {weights / GIB:6.1f} GiB   "
          f"GPUs for the weights alone: {math.ceil(weights / L4_USABLE)}")

FP16   weights  131.4 GiB   GPUs for the weights alone: 7
FP8    weights   65.7 GiB   GPUs for the weights alone: 4
4-bit  weights   32.9 GiB   GPUs for the weights alone: 2


**What you should see:** FP16 needs 7 L4s for the weights alone, FP8 needs 4, and 4-bit needs 2. Even the smallest format does not fit one card.

### Step 3: the one-GPU attempt
This is the version without parallelism: put all the weights on one card and check what is left. The function returns the room left for the cache, or raises an error.

In [3]:
def load_on_one_gpu(weights_bytes):
    """All weights on one L4. Returns the room left for the KV cache, or raises."""
    room = L4_USABLE - weights_bytes
    if room < 0:
        raise ValueError(f"weights need {-room / GIB:.1f} GiB more than one L4 can use")
    return room

for fmt, b in BYTES_PER_PARAM.items():
    try:
        room = load_on_one_gpu(PARAMS_70B * b)
        print(f"{fmt}: fits, {room / GIB:.1f} GiB left for the cache")
    except ValueError as err:
        print(f"{fmt}: does not fit, {err}")

FP16: does not fit, weights need 110.9 GiB more than one L4 can use
FP8: does not fit, weights need 45.1 GiB more than one L4 can use
4-bit: does not fit, weights need 12.3 GiB more than one L4 can use


**What you should see:** all three formats fail. FP16 is short by 110.9 GiB, and even 4-bit is short by 12.3 GiB. This is the check that stops a load on one card.

In [4]:
load_on_one_gpu(PARAMS_70B * 2)   # the engine's load step, FP16, one L4

ValueError: weights need 110.9 GiB more than one L4 can use

### Step 4: the cache for one request
The Brightside request has a 1,600-token prompt and a 200-token answer. It needs this much cache at the end of its answer:

In [5]:
REQUEST_TOKENS = 1_600 + 200
print(f"one Brightside request: {REQUEST_TOKENS * KV_PER_TOKEN / 2**20:.1f} MiB of KV cache")

one Brightside request: 562.5 MiB of KV cache


**What you should see:** 562.5 MiB. That is small next to the weights. The weights decide whether the model runs at all, so they set the first split.

### Step 5: the split (with parallelism)
`plan` splits the weights over `tp` GPUs. Each GPU holds `1/tp` of the weights and `1/tp` of the KV cache. The function rejects a degree that does not divide the head counts, and it reports how many tokens of cache the whole group can hold.

**Why the degree must divide the heads.** A **head** is one separate attention computation inside a layer (lesson 04). The 70B model has 64 query heads and 8 KV heads. A GPU gets whole heads, never half of one, so each share must be a whole number. Over 3 GPUs, 8 KV heads give 2 each with 2 left over, and no GPU can take them. The planner refuses TP=3. Degrees 1, 2, 4 and 8 divide both counts exactly.

In [6]:
def plan(bytes_per_param, tp):
    """Tensor-parallel split: weights per GPU, room left per GPU, and KV tokens for the group."""
    if Q_HEADS % tp or KV_HEADS % tp:
        raise ValueError(f"degree {tp} must divide {Q_HEADS} query heads and {KV_HEADS} KV heads")
    weights_per_gpu = PARAMS_70B * bytes_per_param / tp
    room = L4_USABLE - weights_per_gpu
    kv_tokens = room * tp / KV_PER_TOKEN if room > 0 else 0
    return weights_per_gpu, room, kv_tokens

for fmt, b in BYTES_PER_PARAM.items():
    cells = []
    for tp in (1, 2, 4, 8):
        w, room, toks = plan(b, tp)
        if room > 0:
            cells.append(f"TP={tp} fits, room {room / GIB:.1f} GiB, {toks:,.0f} tokens")
        else:
            cells.append(f"TP={tp} no")
    print(f"{fmt:6} " + " | ".join(cells))

FP16   TP=1 no | TP=2 no | TP=4 no | TP=8 fits, room 4.1 GiB, 108,437 tokens
FP8    TP=1 no | TP=2 no | TP=4 fits, room 4.1 GiB, 54,218 tokens | TP=8 fits, room 12.4 GiB, 323,750 tokens
4-bit  TP=1 no | TP=2 fits, room 4.1 GiB, 27,109 tokens | TP=4 fits, room 12.4 GiB, 161,875 tokens | TP=8 fits, room 16.5 GiB, 431,406 tokens


**What you should see:** one line per format, with every degree in order. FP16 fits only at TP=8, with 4.1 GiB of room per GPU and 108,437 cache tokens for the group. FP8 fits from TP=4 (4.1 GiB room, 54,218 tokens) and 4-bit from TP=2 (4.1 GiB room, 27,109 tokens). Quantization lowers the number of GPUs you need, but it also changes how much cache room is left.

The 108,437 tokens are an **upper bound**. They count only the weights and the cache. The engine also reserves memory for activations and the CUDA workspace, so the real cache is smaller by that memory, an amount this lesson does not model. The exact number is the one the engine logs when it starts, and that is the number to plan with. At 1,800 tokens per request, the upper bound is about 60 requests in flight on the whole group.

### Step 6: the smallest split that fits

In [7]:
for fmt, b in BYTES_PER_PARAM.items():
    tp = next(t for t in (1, 2, 4, 8) if plan(b, t)[1] > 0)
    print(f"{fmt:6} smallest split that fits: TP={tp}")

FP16   smallest split that fits: TP=8
FP8    smallest split that fits: TP=4
4-bit  smallest split that fits: TP=2


**What you should see:** FP16 needs TP=8, FP8 needs TP=4, and 4-bit needs TP=2. The engine does the sharding itself. You set one number in the server command. The command is shown here but not run, because it needs GPUs. Lesson 16 lists the flags, and lesson 22 runs this split on a GPU host:

```bash
vllm serve meta-llama/Llama-3.1-70B-Instruct --tensor-parallel-size 8 --gpu-memory-utilization 0.92
```

Check the flag names with `vllm serve --help=all` on the version you install.

### Step 7: would a bigger single GPU help?
The H200 has 141 GB of memory (NVIDIA's H200 page). Use the same 0.92 utilisation rule. The running example stays on eight 24 GB L4s, which link to each other over PCIe with no NVLink. A PCIe link is slower than NVLink, so the all-reduce cost in lesson 22 is higher than it would be on NVLink cards. Lesson 24 compares the cards.

In [8]:
H200_BYTES = 141e9
print(f"one H200 usable: {H200_BYTES * UTIL / GIB:.1f} GiB, FP16 weights: {PARAMS_70B * 2 / GIB:.1f} GiB")

one H200 usable: 120.8 GiB, FP16 weights: 131.4 GiB


**What you should see:** 120.8 GiB against 131.4 GiB. Even a 141 GB H200 does not hold the FP16 70B model at vLLM's default utilisation.

## How it simplifies the architecture
Without parallelism, a 70B model does not run on one L4 at any precision here (FP16, FP8 or 4-bit all fail in step 3). Your only other options are a much bigger card or a smaller model.

With tensor parallelism, the eight GPUs act as one engine behind one endpoint. Your service still sends one request. The sharding and the all-reduce run inside the engine.

| | Without parallelism | With tensor parallelism (TP = 8) |
|---|---|---|
| GPUs per model copy | 1 (does not fit) | 8 |
| Weights per GPU (FP16) | 131 GiB | 16.4 GiB |
| Cache room for the group | none | about 108,000 tokens |
| Your code | one endpoint | one endpoint (same) |
| New cost | none, because it fails | an all-reduce per layer over the GPU link |


### What tensor parallelism does not solve
- **Communication.** A layer has two all-reduces, one after attention and one after the MLP. Over 80 layers that is 160 all-reduces per forward pass, so every forward pass waits for 160 of them. An L4 talks to its neighbours over PCIe, not NVLink, so this cost is real. Lesson 22 measures it.
- **Throughput.** Eight GPUs give one copy of the model. More copies, called **replicas**, give more requests per second (lesson 29; the load maths is in lesson 26).
- **Quality.** Lower precision saves GPUs, but it can cost accuracy (lesson 20).

Alternatives: quantize first (lesson 20), split by layers across GPUs with pipeline parallelism (lesson 23), or use a single bigger GPU. Step 7 shows that an H200 still does not hold the FP16 weights at 0.92 utilisation, so a bigger card helps only with smaller weights, and the split may still be needed for the cache.

## At work
- **Cloudflare (May 2026)** says Kimi K2.5, with about 560 GB of weights, needs 8 H100 GPUs. The weights alone set the minimum group size. → lesson 23.
- **Meta and PyTorch (Sep 2025)** ran disaggregated serving on 8-H100 hosts. Over TCP, time to first token (TTFT) regressed sharply under heavy load because the network was the bottleneck: moving data between GPUs is a real cost. → lesson 34.
- **vLLM's issue tracker** has more than 5,800 issues that mention tensor parallelism (a keyword count, an upper bound), so developers meet this split often. → lesson 22.

Sources: Cloudflare, "High-performance LLMs" (May 2026): https://blog.cloudflare.com/high-performance-llms/ · Meta and PyTorch, "Disaggregated Inference at Scale with PyTorch & vLLM" (2025-09-12): https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/ · vLLM issues: https://github.com/vllm-project/vllm/issues

## Common mistakes
**1. Dividing the 70B model by a degree that does not divide the heads.** Three GPUs can look like a convenient number, but 64 query heads do not split into three equal parts. The planner refuses this.

In [9]:
plan(BYTES_PER_PARAM["FP8"], 3)   # TP=3: 64 heads cannot be split in three

ValueError: degree 3 must divide 64 query heads and 8 KV heads

The fix is to use a degree that divides 64 and 8: 1, 2, 4 or 8.

**2. Counting only the weights.** Eight L4s hold the FP16 weights, but they leave only 4.1 GiB each for the cache, activations and the CUDA workspace. Count the room left after the weights, not only the fit.

## Try it
Your team has six L4s. Can you run the 4-bit model with TP=6? If not, what is the largest TP you can use, and what room per GPU does it leave? Use `plan` and the head counts from step 5.

In [10]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [11]:
try:
    plan(BYTES_PER_PARAM["4-bit"], 6)
except ValueError as err:
    print(f"TP=6 refused: {err}")

best = max(t for t in (1, 2, 4, 8) if t <= 6 and Q_HEADS % t == 0 and KV_HEADS % t == 0)
w, room, toks = plan(BYTES_PER_PARAM["4-bit"], best)
print(f"largest valid TP on six GPUs: {best}; 4-bit weights per GPU {w / GIB:.1f} GiB, room {room / GIB:.2f} GiB")

TP=6 refused: degree 6 must divide 64 query heads and 8 KV heads
largest valid TP on six GPUs: 4; 4-bit weights per GPU 8.2 GiB, room 12.35 GiB


## Recap
- A 70B model has about 70.6 billion parameters: 141 GB in FP16 and 70 GB in FP8.
- One L4 gives vLLM 20.56 GiB (24 GB × 0.92). The FP16 weights alone need 7 of them.
- Tensor parallelism shards the weights over TP GPUs. The degree must divide the 64 query heads and 8 KV heads.
- FP16 needs TP=8, FP8 needs TP=4 and 4-bit needs TP=2. The price is two all-reduces per layer, 160 per forward pass.

**Interview:** "How many GPUs does a 70B model in FP16 need?" Strong answer: 70.6 billion parameters × 2 bytes = 141 GB of weights. An L4 gives the engine about 20.6 GiB, so the weights alone need 7. The degree must divide 64 and 8, so the group is 8 GPUs. Then name the cost: an all-reduce on every layer.